# 분석파이프라인만들기_타이타닉데이터셋_train - 중요!!
* train데이터로 훈련했을 때 사용한 label encoding, scaler, model

### 타이타닉 데이터로 “생존 여부 예측 모델”을 만드는 전체 과정 정리

### 타이타닉 생존 예측 모델을 파이프라인으로 깔끔하게 만드는 실습 코드

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib

In [2]:
df = pd.read_csv("./data/Titanic_train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 이전에 확인했던 데이터 특성과, 파생변수, 데이터처리 방법을 이용해 파이프라인 구축

In [3]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [4]:
data = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp',
       'Parch', 'Embarked']]
data

,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
0,0,3,male,22.0,1,0,S
1,1,1,female,38.0,1,0,C
2,1,3,female,26.0,0,0,S
3,1,1,female,35.0,1,0,S
4,0,3,male,35.0,0,0,S
...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,S
887,1,1,female,19.0,0,0,S
888,0,3,female,NaN,1,2,S
889,1,1,male,26.0,0,0,C


In [5]:
from sklearn.model_selection import train_test_split

In [6]:
train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data['Survived'])

In [7]:
train

,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
692,1,3,male,NaN,0,0,S
481,0,2,male,NaN,0,0,S
527,0,1,male,NaN,0,0,S
855,1,3,female,18.0,0,1,S
801,1,2,female,31.0,1,1,S
...,...,...,...,...,...,...,...
359,1,3,female,NaN,0,0,Q
258,1,1,female,35.0,0,0,C
736,0,3,female,48.0,1,3,S
462,0,1,male,47.0,0,0,S


In [8]:
test

,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
565,0,3,male,24.0,2,0,S
160,0,3,male,44.0,0,1,S
553,1,3,male,22.0,0,0,C
860,0,3,male,41.0,2,0,S
241,1,3,female,NaN,1,0,Q
...,...,...,...,...,...,...,...
880,1,2,female,25.0,0,1,S
91,0,3,male,20.0,0,0,S
883,0,2,male,28.0,0,0,S
473,1,2,female,23.0,0,0,C


In [9]:
print(train['Survived'].value_counts())
print(test['Survived'].value_counts())

Survived
0    439
1    273
Name: count, dtype: int64
Survived
0    110
1     69
Name: count, dtype: int64


In [10]:
test.to_csv("./data/titanic_test.csv", index=False)

In [11]:
train['family'] = train['SibSp'] + train['Parch']
train = train.drop(['SibSp','Parch'], axis=1)
train

,Survived,Pclass,Sex,Age,Embarked,family
692,1,3,male,NaN,S,0
481,0,2,male,NaN,S,0
527,0,1,male,NaN,S,0
855,1,3,female,18.0,S,1
801,1,2,female,31.0,S,2
...,...,...,...,...,...,...
359,1,3,female,NaN,Q,0
258,1,1,female,35.0,C,0
736,0,3,female,48.0,S,4
462,0,1,male,47.0,S,0


In [12]:
# X / y 나누기
# X = 문제, y = 정답

X = train.drop(['Survived'], axis=1)
y = train['Survived']

In [13]:
X

,Pclass,Sex,Age,Embarked,family
692,3,male,NaN,S,0
481,2,male,NaN,S,0
527,1,male,NaN,S,0
855,3,female,18.0,S,1
801,2,female,31.0,S,2
...,...,...,...,...,...
359,3,female,NaN,Q,0
258,1,female,35.0,C,0
736,3,female,48.0,S,4
462,1,male,47.0,S,0


In [14]:
y

692    1
481    0
527    0
855    1
801    1
      ..
359    1
258    1
736    0
462    0
507    1
Name: Survived, Length: 712, dtype: int64

# 범주형(문자형) 컬럼과 숫자형 컬럼 분리
### 다른 변수에 담아서 따로 저장

처리 방법이 다르기 때문

- 숫자 → 평균 채우기 + 스케일링

- 문자 → 최빈값 + 원핫인코딩

In [15]:
cat_cols = X.select_dtypes(include='object').columns
cat_cols

C:\Users\a\AppData\Local\Temp\ipykernel_14108\2489625261.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns


Index(['Sex', 'Embarked'], dtype='str')

In [16]:
num_cols = X.select_dtypes(exclude='object').columns
num_cols

Index(['Pclass', 'Age', 'family'], dtype='str')

# X, y 를 train/valid로 나누기

In [17]:
len(X)

712

In [18]:
# X_train → 공부, X_valid → 시험

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 데이터 전처리 파이프 구성
* 수치형 자료: 평균으로 대체 + 스케일링
* 범주형 자료: 최빈값 대체 + 원-핫 인코딩

## 파이프라인 만들기

파이프라인이 뭐냐면:

전처리 + 모델을 한 번에 묶는 것

### 1) 전처리 구성
1. 결측값 채우기
2. 문자를 숫자로 바꾸기


✔ 숫자형 데이터 (Age, family 등)

결측값 → 평균으로 채움

스케일링 (값 크기 맞춤)

✔ 문자형 데이터 (Sex, Embarked)

결측값 → 최빈값으로 채움

원핫인코딩 (남/여를 0,1 같은 숫자로 변환)

### 2) 모델 여러 개 돌려보기

DecisionTree

RandomForest

(XGBoost도 시도하려 한 듯)

그리고

👉 RandomizedSearchCV로
👉 하이퍼파라미터 자동으로 여러 번 돌려봄


---

# 📦 Pipeline 사용법 아주 자세한 설명

## 1️⃣ Pipeline은 무엇을 하는가?

Pipeline은 여러 단계를 **순서대로 연결한 하나의 모델 객체**입니다.

```python
pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", model)
])
```

겉으로 보면 `pipe`는 모델 하나처럼 보이지만,
내부에서는 다음 순서로 동작합니다.

1. preprocess (전처리)
2. model (학습/예측)

---

## 2️⃣ fit()이 호출되면 내부에서 일어나는 일

```python
pipe.fit(X_train, y_train)
```

### 단계별 동작

① preprocess.fit_transform(X_train)

- 수치형 중앙값 계산 (train 기준)
- 평균/표준편차 계산 (train 기준)
- 범주 목록 기억 (train 기준)
- X_train을 숫자 행렬로 변환

② model.fit(X_train_transformed, y_train)

- 변환된 train 데이터로 모델 학습

🔥 핵심: 전처리 통계는 **반드시 train에서만 계산됨**
→ 데이터 누수 방지

---

## 3️⃣ predict()가 호출되면 내부에서 일어나는 일

```python
pipe.predict(X_test)
```

① preprocess.transform(X_test)
- 이미 train에서 학습된 통계값 사용
- test로는 절대 fit하지 않음

② model.predict(X_test_transformed)

🔥 핵심: test는 transform만 수행

---

## 4️⃣ RandomizedSearchCV와 Pipeline

파라미터 이름 규칙:

```
스텝이름__파라미터
```

예시:

```
model__max_depth
model__C
preprocess__cat__ohe__handle_unknown
```

언더바 2개(__)는 “단계 안으로 들어간다”는 의미입니다.

---

## 5️⃣ 왜 Pipeline이 실무에서 중요한가?

✔ 데이터 누수 방지  
✔ 교차검증에서도 fold별로 안전하게 전처리  
✔ 모델 저장 시 전처리 포함 저장 가능  
✔ 서비스에서 raw 데이터 그대로 predict 가능  

---

## 6️⃣ ColumnTransformer + Pipeline 구조 정리

```
원본 데이터
   ↓
ColumnTransformer
   ├─ 수치형: Imputer → Scaler
   └─ 범주형: Imputer → OneHot
   ↓
모델
```

이 전체 공정을 하나로 묶은 것이 Pipeline입니다.


In [19]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [20]:
num_cols

Index(['Pclass', 'Age', 'family'], dtype='str')

In [24]:
si = SimpleImputer(strategy='mean')
si.fit(X_train['Age'])
si

ValueError: Expected a 2-dimensional container but got <class 'pandas.Series'> instead. Pass a DataFrame containing a single row (i.e. single sample) or a single column (i.e. single feature) instead.

# 가장 먼저 -> 전처리 단계


# 숫자와 문자 데이터를 각각 다르게 처리해서 모델에 넣을 준비를 하는 자동 전처리 공장

### ("scaler", StandardScaler()) : 숫자 크기 맞추기

키는 170,
요금은 50000,
가족수는 3

→ 숫자 크기 차이 심함
→ 모델이 혼란스러워함

그래서 평균 0, 표준편차 1로 맞춰줌.

### OneHotEncoder : 문자를 숫자로 바꾸는 기계

예:
Sex = male, female

→ male = 1 / female = 0

- drop='first'

이건 왜 쓰냐면

카테고리가 2개면 하나만 있어도 충분해.

예:
male / female

female이 0이면
male은 자동으로 1임.

그래서 하나 삭제해서 중복 방지.

- handle_unknown='ignore'

test 데이터에
train 때 없던 값이 나오면

👉 에러 내지 말고 무시해라

라는 뜻.

실전에서 매우 중요.

### ColumnTransformer

자동 분류기

숫자는 num_pipe에 넣고, 문자는 cat_pipe에 넣어라

- remainder="drop" : 우리가 지정하지 않은 컬럼은 버려라



In [25]:
# 숫자 데이터 전용 처리
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='mean')), # 결측값을 평균으로 채움
    ("scaler", StandardScaler()) # 숫자 크기 맞추기
]) 

# 문자 전용
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='most_frequent')),
    # 결측값을 가장 많이 나온 값으로 채움
    ("ohe", OneHotEncoder(drop='first', handle_unknown='ignore'))
    # OneHotEncoder : 문자를 숫자로 바꾸는 기계
    
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ],
    remainder="drop"
)

In [26]:
print(preprocess)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 Index(['Pclass', 'Age', 'family'], dtype='str')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'))]),
                                 Index(['Sex', 'Embarked'], dtype='str'))])


지금까지 한 것 👇

1️⃣ 데이터 정리
2️⃣ 전처리 파이프 만들기
3️⃣ Pipeline 구조 완성

👉 이건 요리 준비 끝난 상태

# 지금까지 전처리 끝 → 어떤 모델이 제일 좋은지 대결시키는 단계해야 함

# <모델 정의, 하이퍼파라미터 정의, randomsearch>

### 하이퍼파라미터 : 모델을 학습시키기 전에 우리가 직접 정해줘야 하는 설정값
- 1️⃣ DecisionTree
max_depth = 3

👉 나무를 몇 층까지 만들지?

깊게 만들면 → 복잡해짐 (과적합 위험)

얕게 만들면 → 단순함 (성능 낮을 수도)

이 max_depth가 하이퍼파라미터야.

- 2️⃣ RandomForest
n_estimators = 500

👉 나무를 몇 개 만들지?

이것도 우리가 정해줘야 하니까 하이퍼파라미터.

- 3️⃣ LogisticRegression
C = 1.0

👉 규제를 얼마나 줄지?

이것도 하이퍼파라미터.

🔹 파라미터

→ 모델이 학습하면서 자동으로 배우는 값
예: 가중치(weight)

🔹 하이퍼파라미터

→ 우리가 직접 정하는 값
예: max_depth, C, n_estimators

## 📌 왜 하이퍼파라미터가 중요하냐?

같은 데이터라도

max_depth = 2 → 성능 0.75

max_depth = 10 → 성능 0.83

max_depth = 30 → 과적합

이렇게 달라질 수 있어.

그래서 여러 개 시험 보는 게

### 👉 RandomizedSearchCV

In [27]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
f1_score, roc_auc_score, classification_report)

## 학습하는 거 아님 모델 후보랑 경기 규칙 정하는 코드!!
### 1️⃣ cv = StratifiedKFold(...)
- cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
📌 이건 경기 방식 정하는 거야.
✅ n_splits=5

데이터를 5등분해서
5번 시험 보겠다는 뜻.

✅ Stratified

이게 핵심.

타이타닉은

0 = 사망

1 = 생존

비율이 완전 50:50이 아님.

Stratified는
👉 각 조각에도 생존/사망 비율을 비슷하게 유지해줌.
그래야 공정한 시험이 됨.

✅ shuffle=True

데이터를 섞고 나눈다.

✅ random_state=42

항상 같은 방식으로 나누기 위해 고정값 설정.

### 2️⃣ scoring = "roc_auc"
scoring = "roc_auc"

이건

“어떤 점수로 모델을 평가할지” 정하는 거야.

### 3️⃣ models = { ... }

이제 선수 명단 나옴 👇
“시험 볼 모델 후보들 모음”

In [28]:
# 이 코드는 학습하는 거 아니고
# 모델 대결을 위한 경기 규칙 + 선수 명단 정하기

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = "roc_auc"

models = {
    "DecisionTree" : DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "RandomForest" : RandomForestClassifier(random_state=42, class_weight="balanced"),
    "LogisticRegression" : LogisticRegression(max_iter=2000, class_weight="balanced"),
    "SVM" : SVC(probability=True, class_weight="balanced", random_state=42),
    "XGBoost" : XGBClassifier(random_state=42)
}

In [29]:
models

{'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
 'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42),
 'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=2000),
 'SVM': SVC(class_weight='balanced', probability=True, random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraint

# 모델별 하이퍼 파라미터 정의

In [30]:
XGBClassifier(learn)

NameError: name 'learn' is not defined

### params = 시험 문제 후보지
### RandomizedSearchCV = 랜덤으로 문제 뽑아서 시험 보는 사람

### 💡 model__ 왜 붙냐?

Pipeline 구조가:

pipeline
   ├ preprocessor
   └ model

그래서

model__max_depth

= model 안에 있는 max_depth 건드리겠다

라는 뜻이야.

In [31]:
# 우리가 시험할 여러 하이퍼파라미터 값들의 후보를 적어두고
# RandomizedSearchCV가 그 중 일부를 뽑아서 시험 보는 거야.

params = {
    "DecisionTree" : {
        "model__max_depth" : [2, 3, 4, 5, 7, 10, 15, 20],
        "model__min_samples_split" : [2,5, 10, 20, 30],
        "model__min_samples_leaf" : [1,2,5, 10, 20],
        "model__criterion" : ["gini", "entropy", "log_loss"]
        
    },
    "RandomForest" : {
        "model__n_estimators" : [200, 400, 600, 800, 1000],
        "model__criterion" : ["gini", "entropy", "log_loss"],
        "model__min_samples_split" : [2,5, 10, 20, 30],
        "model__min_samples_leaf" : [1,2,5, 10, 20],
        "model__max_depth" : [2, 3, 4, 5, 7, 10, 15, 20]
    },
    "LogisticRegression" : {
        "model__C" : np.logspace(-3, 2, 50),
        "model__solver" : ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
        "model__penalty" : ['l1', 'l2', 'elasticnet']
    },
    "SVM" : {
        "model__C" : np.logspace(-3, 2, 50),
        "model__gamma" : ['scale', 'auto'] + list(np.logspace(-4, 0, 10)),
        "model__kernel" : ['rbf', 'sigmoid']
    }, 
    "XGBoost" : {
        "model__n_estimators" : [200, 400, 600, 800, 1000],
        "model__max_depth" : [2, 3, 4, 5, 7, 10, 15, 20],
        "model__learning_rate" : list(np.logspace(-3, -0.3, 20))
    } 
}

params.keys()

dict_keys(['DecisionTree', 'RandomForest', 'LogisticRegression', 'SVM', 'XGBoost'])

지금까지는
👉 “어떤 파라미터가 제일 좋냐?”를 찾는 단계였고

아래 함수는
👉 “그래서 테스트셋에서 실제 성능이 얼마냐?”를 보는 단계야.

In [32]:
# 튜닝 끝난 모델을 실제 테스트 데이터에서 평가하기 위해 미리 만들어 둔 함수
# 실행할 때 아래에서 쓸 evaluate_on_test함수 미리 정의해둠

from time import time

def evaluate_on_test(best_estimator, X_test, y_test):
    pred = best_estimator.predict(X_test)

    score = None
    if hasattr(best_estimator, "predict_proba"):
        score = best_estimator.predict_proba(X_test)[:, 1]
    elif hasattr(best_estimator, "decision_function"):
        score = best_estimator.decision_function(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    }

    if score is not None:
        metrics["roc_auc"] = roc_auc_score(y_test, score)

    else:
        metrics["roc_auc"] = np.nan


    return metrics

### 아래 셀이 진짜로 다 실행하는 핵심 셀
지금까지는 준비 단계였고,
이 셀에서 처음으로:

💥 튜닝 + 학습 + 평가 + 모델 비교
전부 실제로 실행됨.

👉 모든 모델을 자동으로 튜닝하고 <br>
👉 각 모델의 최고 성능을 비교하는 셀

models 딕셔너리에 있는:

DecisionTree

RandomForest

LogisticRegression

SVM

XGBoost

하나씩 자동 실행함.

## 중요!! RandomizedSearchCV 생성
### params에서 하이퍼파라미터 조합을 랜덤으로 뽑아서, 교차검증으로 성능 비교하고, 가장 좋은 모델을 자동으로 선택
```python
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params[name], 해당 모델 파라미터 후보 가져옴
        n_iter=N_ITER,
        scoring=scoring,
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        refit=True)
  ```


### 결과 정리
``` python
row = {
    "model": name,
    "best_cv_score(roc-auc)": search.best_score_,
    **test_metrics,
    "best_params": search.best_params_
}
```
교차검증 점수, 테스트 점수, 최적 파라미터

전부 저장

In [33]:
results = [] # → 각 모델 결과 저장용
best_search_objects = {} # → 각 모델의 search 객체 저장

N_ITER = 30  # → 랜덤 조합 30개만 시험
RANDOM_STATE = 42 # → 랜덤 고정

for name, model in models.items(): # 모델 하나씩 반복
#     print(name, model)
    print("=" * 30, name, "=" * 30)
    
    # 파이프라인 생성 (전처리 preprocess + 모델 model을 하나로 묶음)
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])
    
    # 중요!! RandomizedSearchCV 생성
    # params에서 하이퍼파라미터 조합을 랜덤으로 뽑아서, 교차검증으로 성능 비교하고, 가장 좋은 모델을 자동으로 선택
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params[name], # → 해당 모델 파라미터 후보 가져옴
        n_iter=N_ITER,
        scoring=scoring,
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        refit=True)
    
    # 실제 튜닝 시작
    search.fit(X_train, y_train) # 위의 search 실행시킴
    best_est = search.best_estimator_
    # = search가 여러 하이퍼파라미터 조합을 시험해본 뒤 그중 제일 성능 좋았던 모델을 꺼낸다는 뜻
    test_metrics = evaluate_on_test(best_est, X_valid, y_valid) # 함수 이용-테스트 데이터 평가
    
    # 결과 정리
    row = {
        "model" : name,
        "best_cv_score(roc-auc)" : search.best_score_,
        **test_metrics,
        "best_params" : search.best_params_
    }
    results.append(row)
    best_search_objects[name] = search
    
    print(f"best_cv_roc_auc = {search.best_score_:.4f} | test roc-auc = {test_metrics['roc_auc']:.4f}")
    
results_df = pd.DataFrame(results).sort_values(by='roc_auc', ascending=False)
results_df

============================== DecisionTree ==============================
best_cv_roc_auc = 0.8591 | test roc-auc = 0.8556
============================== RandomForest ==============================
best_cv_roc_auc = 0.8736 | test roc-auc = 0.8405
============================== LogisticRegression ==============================


C:\Users\a\miniforge3\envs\fintech\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
100 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\a\miniforge3\envs\fintech\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\a\miniforge3\envs\fintech\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\a\miniforge3\envs\fintech\Lib\site-packages\sklearn\pipeline.py", line 

best_cv_roc_auc = 0.8579 | test roc-auc = 0.8289
============================== SVM ==============================
best_cv_roc_auc = 0.8620 | test roc-auc = 0.8256
============================== XGBoost ==============================
best_cv_roc_auc = 0.8731 | test roc-auc = 0.8720


,model,best_cv_score(roc-auc),accuracy,precision,recall,f1,roc_auc,best_params
4,XGBoost,0.873056,0.839161,0.863636,0.690909,0.767677,0.872004,"{'model__n_estimators': 1000, 'model__max_dept..."
0,DecisionTree,0.859101,0.790210,0.692308,0.818182,0.750000,0.855579,"{'model__min_samples_split': 30, 'model__min_s..."
1,RandomForest,0.873568,0.811189,0.759259,0.745455,0.752294,0.840496,"{'model__n_estimators': 1000, 'model__min_samp..."
2,LogisticRegression,0.857920,0.797203,0.709677,0.800000,0.752137,0.828926,"{'model__solver': 'newton-cg', 'model__penalty..."
3,SVM,0.862000,0.804196,0.764706,0.709091,0.735849,0.825620,"{'model__kernel': 'rbf', 'model__gamma': 0.129..."


In [34]:
results_df['best_params'][0] # 각 모델에서 찾은 “최적 하이퍼파라미터 조합”에서 1등

{'model__n_estimators': 1000,
 'model__max_depth': 2,
 'model__learning_rate': np.float64(0.007122485584859914)}

In [35]:
# search = RandomizedSearchCV(...) -> RandomizedSearchCV 객체
# 이 튜닝 기계가 어떻게 설정되어 있는지
print(search)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocess',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer()),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               Index(['Pclass', 'Age', 'family'], dtype='str')),
                                                                              ('cat',
                                                                               Pipeline(steps=[('imputer',
      

- best_estimator_ : 우리가 estimator=pipe로 넣었기 때문에 -> 전처리 + 모델이 합쳐진 Pipeline 전체


In [36]:
# 튜닝 끝난 후, 가장 성능 좋았던 최종 모델의 전체 구조를 출력
print(search.best_estimator_)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Pclass', 'Age', 'family'], dtype='str')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(drop='first',
                                                                       

# best 모델 저장하기

# 변수/모델/스케일러 등을 저장/로드 하는 방법 joblib(직렬화)
* joblib.dump(변수명, 경로 파일명)
* 변수명 = joblib.load(경로 파일명)

In [37]:
results_df

# 모델 별로 다 해봄
# roc_auc로 판단해봤을 때 XGBoost가 가장 효과가 좋음

,model,best_cv_score(roc-auc),accuracy,precision,recall,f1,roc_auc,best_params
4,XGBoost,0.873056,0.839161,0.863636,0.690909,0.767677,0.872004,"{'model__n_estimators': 1000, 'model__max_dept..."
0,DecisionTree,0.859101,0.790210,0.692308,0.818182,0.750000,0.855579,"{'model__min_samples_split': 30, 'model__min_s..."
1,RandomForest,0.873568,0.811189,0.759259,0.745455,0.752294,0.840496,"{'model__n_estimators': 1000, 'model__min_samp..."
2,LogisticRegression,0.857920,0.797203,0.709677,0.800000,0.752137,0.828926,"{'model__solver': 'newton-cg', 'model__penalty..."
3,SVM,0.862000,0.804196,0.764706,0.709091,0.735849,0.825620,"{'model__kernel': 'rbf', 'model__gamma': 0.129..."


In [38]:
# 모델을 변수에 담음
best_pipe = search.best_estimator_

- joblib : 파이썬 객체를 그대로 파일로 저장할 수 있게 해주는 라이브러리

In [39]:
import joblib

In [40]:
# joblib을 이용해서 모델 저장함
joblib.dump(best_pipe, "./data/titanic_best_model.joblib")

['./data/titanic_best_model.joblib']

In [41]:
print(best_pipe)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Pclass', 'Age', 'family'], dtype='str')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(drop='first',
                                                                       